# Candidate Recovery Metric — Resting Heart Rate

Resting heart rate is the wearable signal least dependent on a patient choosing to be active, so
**it can register recovery in someone too unwell to walk — exactly the patients a step-count
outcome loses**. It is also the hardest to obtain: All of Us publishes daily heart-rate zone
summaries but no resting value, so the metric has to be derived from the minute-level table. This
notebook audits coverage from the cheap daily summary first and only then prices what deriving a
resting value would cost, because cost is a real selection criterion here rather than an
afterthought.

**Governance.** Every query aggregates inside BigQuery and returns counts, quantiles or
correlations only, so no participant-level rows enter the notebook. Unsuppressed figures stay
git-ignored while `*_suppressed.png` is committable, and the small-cell threshold must be
confirmed against current All of Us policy before anything is published.

## Shared Imports and Methods

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored figure
and cache directories, and builds the cohort plus one index date per patient. **The
metric-specific CTE and the query budget are both defined here**, so the SQL this notebook runs
and the ceiling it refuses to spend past sit in one place.

#### Imports, Cohort SQL and the Query Budget

The budget exists because the minute-level heart-rate table is the one object in this project
large enough for a careless scan to be expensive. Everything downstream of it is priced by a dry
run before it executes.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("fitbit_heart_rate")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

# Days a patient needs in EACH window to be analysable. Part 1 counts these patients
# and Part 2 derives the resting value for exactly them, so the rule is written once:
# a second copy of the 7 is how a curve and the scorecard row behind it drift apart.
PAIRED_MIN_DAYS = 7

_window_case = srp_common.window_case_sql("rel_day")

METRIC_CTE = f"""
    hr_daily AS (
        SELECT
            h.person_id,
            DATE_DIFF(h.date, i.index_date, DAY) AS rel_day,
            SUM(h.minute_in_zone)                AS minutes_recorded
        FROM `{CDR}`.heart_rate_summary AS h
        JOIN index_surgery AS i ON i.person_id = h.person_id
        GROUP BY h.person_id, rel_day
    ),
    hr_window_counts AS (
        SELECT
            person_id,
            {_window_case} AS window_name,
            COUNT(DISTINCT rel_day) AS n_days
        FROM hr_daily
        WHERE {_window_case} IS NOT NULL
        GROUP BY person_id, window_name
    ),
    hr_window_days AS (
        SELECT
            person_id,
            MAX(IF(window_name = 'baseline', n_days, 0)) AS baseline_days,
            MAX(IF(window_name = 'postop',   n_days, 0)) AS postop_days
        FROM hr_window_counts
        GROUP BY person_id
    ),
    paired_hr_person AS (
        SELECT person_id
        FROM hr_window_days
        WHERE baseline_days >= {PAIRED_MIN_DAYS}
          AND postop_days   >= {PAIRED_MIN_DAYS}
    )"""

# Deriving a resting value needs the minute-level table, which is orders of magnitude
# larger. Nothing reads it until a dry run has priced the scan and it comes in under
# this budget -- an accidental full scan is the expensive mistake in this repo.
MINUTE_LEVEL_BUDGET_GIB = 200

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# Fail here, not several minutes into a BigQuery run, if the CDR does not have
# what this notebook reads. cohort_ctes checks its own tables separately.
cfg.require_columns({
    "heart_rate_intraday": ["person_id", "datetime", "heart_rate_value"],
    "heart_rate_summary": ["person_id", "date", "minute_in_zone"],
})

## Part 1 — Coverage From the Cheap Table

Coverage is settled entirely from the daily summary, which is small enough to scan across the
whole relative-day range, so the expensive table is never touched to answer a question the cheap
one can answer. **Coverage here is an upper bound on resting heart rate**, which additionally
needs enough minutes in a day to estimate from, and a day counts as covered when the summary
records any minutes at all — generous on purpose, so the bound is honest.

#### Who Is Covered, and Who Has Both Windows

The daily summary and the minute-level table are named and shaped differently across All of Us
releases, so both are declared to `require_columns` and validated at `init()` against
`cdr_schema.py` rather than probed at runtime. **The paired count is the binding constraint on any
return-to-baseline construction**, as it is for steps — neither window alone decides anything.

In [ ]:
# Coverage from the daily summary alone. This is an UPPER BOUND on resting heart rate,
# which additionally needs enough minutes in a day to estimate from.
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                       AS cohort_patients,
        (SELECT COUNT(DISTINCT person_id) FROM hr_daily)    AS patients_with_any_hr,
        (SELECT COUNT(*) FROM hr_daily)                     AS hr_days
"""

cov = cached_gbq(coverage_sql, "heart_rate_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])

print(f"cohort patients          : {cohort_patients:>9,}")
print(f"  ... with >=1 HR day    : {int(cov['patients_with_any_hr']):>9,}"
      f"   ({int(cov['patients_with_any_hr']) / cohort_patients:6.1%} of cohort)")
print(f"heart-rate days          : {int(cov['hr_days']):>9,}")

curve_sql = CTES + """
    SELECT rel_day, COUNT(DISTINCT person_id) AS n_patients
    FROM hr_daily
    WHERE rel_day BETWEEN -180 AND 365
    GROUP BY rel_day
    ORDER BY rel_day
"""

hr_curve = cached_gbq(curve_sql, "heart_rate_coverage_curve")

srp_common.plot_rel_day_curve(
    cfg, hr_curve, "n_patients",
    title="Patients with a Fitbit heart-rate record, by day relative to surgery",
    ylabel="patients with a heart-rate record", suppress=False,
    save_as="heart_rate_coverage")
srp_common.plot_rel_day_curve(
    cfg, hr_curve, "n_patients",
    title="Patients with a Fitbit heart-rate record, by day relative to surgery",
    ylabel="patients with a heart-rate record", suppress=True,
    save_as="heart_rate_coverage")

# The paired count: both windows populated for the same patient. This reads the same
# hr_window_days CTE that paired_hr_person is cut from, so the number printed here and
# the population Part 2 derives its resting value over cannot disagree.
paired_sql = CTES + f"""
    SELECT
        COUNT(*)                                     AS patients_any_window,
        COUNTIF(baseline_days >= {PAIRED_MIN_DAYS})  AS baseline_7d,
        COUNTIF(postop_days   >= {PAIRED_MIN_DAYS})  AS postop_7d,
        COUNTIF(baseline_days >= {PAIRED_MIN_DAYS}
            AND postop_days   >= {PAIRED_MIN_DAYS})  AS paired_7d
    FROM hr_window_days
"""

paired = cached_gbq(paired_sql, "heart_rate_paired_windows").iloc[0]
paired_7d = int(paired["paired_7d"])

for label in ["patients_any_window", "baseline_7d", "postop_7d", "paired_7d"]:
    n = int(paired[label])
    flag = "  << below threshold, do not report" if 0 < n < SUPPRESS_THRESHOLD else ""
    print(f"{label:22s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort){flag}")

## Part 2 — What the Derived Signal Costs, and Whether It Moves

The resting value is taken as each patient-day's 5th percentile of minute-level readings, which is
the usual proxy when a device's own resting figure is not published. **The curve is restricted to
the paired-window patients Part 1 counted** — the same `paired_hr_person` set, read from the shared
CTE rather than re-derived — so the population behind this figure is the one the scorecard calls
analysable, and a movement in it cannot be patients entering and leaving the denominator between
the two windows. **The query runs only if a dry run prices it under the budget set in setup**, so an
accidental full scan of the minute-level table cannot happen here. A metric no surgery visibly
perturbs cannot be a recovery outcome however good its coverage, so cost and signal are asked in
the same breath.

#### Pricing the Minute-Level Derivation and Reading Its Curve

The dry run is not advisory: the query is skipped outright if the estimate exceeds the budget,
and the notebook says so rather than quietly spending. Restricting to the paired set is a
correctness choice rather than a cost one — a join on `person_id` prunes bytes scanned only if the
table is clustered on it, so the estimate may barely move. Quantiles are computed inside BigQuery so
only the per-day summary crosses into the notebook.

In [ ]:
rhr_sql = CTES + f"""
    , minute_hr AS (
        SELECT
            m.person_id,
            DATE_DIFF(DATE(m.datetime), i.index_date, DAY) AS rel_day,
            m.heart_rate_value
        FROM `{CDR}`.heart_rate_intraday AS m
        JOIN index_surgery    AS i ON i.person_id = m.person_id
        JOIN paired_hr_person AS p ON p.person_id = m.person_id
        WHERE DATE(m.datetime) BETWEEN DATE_SUB(i.index_date, INTERVAL 60 DAY)
                                   AND DATE_ADD(i.index_date, INTERVAL 180 DAY)
    ),
    daily_rhr AS (
        SELECT
            person_id,
            rel_day,
            COUNT(*) AS minutes,
            APPROX_QUANTILES(heart_rate_value, 100)[OFFSET(5)] AS resting_hr
        FROM minute_hr
        GROUP BY person_id, rel_day
        HAVING COUNT(*) >= 180
    )
    SELECT
        rel_day,
        COUNT(DISTINCT person_id)                          AS n_patients,
        APPROX_QUANTILES(resting_hr, 100)[OFFSET(25)]      AS p25_rhr,
        APPROX_QUANTILES(resting_hr, 100)[OFFSET(50)]      AS p50_rhr,
        APPROX_QUANTILES(resting_hr, 100)[OFFSET(75)]      AS p75_rhr
    FROM daily_rhr
    GROUP BY rel_day
    ORDER BY rel_day
"""

estimated_bytes = cfg.dry_run_bytes(rhr_sql)
rhr_curve = None

if 0 <= estimated_bytes <= MINUTE_LEVEL_BUDGET_GIB * 2**30:
    rhr_curve = cached_gbq(rhr_sql, "resting_heart_rate_curve")
    srp_common.plot_rel_day_curve(
        cfg, rhr_curve, "p50_rhr", band=("p25_rhr", "p75_rhr"),
        title="Median derived resting heart rate (IQR band), by day relative to surgery",
        ylabel="resting heart rate (bpm)", suppress=False, save_as="resting_heart_rate")
    srp_common.plot_rel_day_curve(
        cfg, rhr_curve, "p50_rhr", band=("p25_rhr", "p75_rhr"),
        title="Median derived resting heart rate (IQR band), by day relative to surgery",
        ylabel="resting heart rate (bpm)", suppress=True, save_as="resting_heart_rate")
elif estimated_bytes < 0:
    print("Dry run unavailable, so the scan is unpriced and the query was NOT run. "
          "Price it by hand before raising MINUTE_LEVEL_BUDGET_GIB or running it directly.")
else:
    print(f"Estimated scan {estimated_bytes / 2**30:,.1f} GiB exceeds the "
          f"{MINUTE_LEVEL_BUDGET_GIB} GiB budget -- narrow the day range or raise the budget "
          "deliberately. The cost itself is an argument against this metric.")

## Part 3 — Verdict

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. **Nothing written here is a
participant-level value**: the row is counts, fractions and hand-entered judgements.

#### The Scorecard Row

Writing the verdict as data rather than prose is what lets the cross-metric comparison be
computed instead of transcribed. The hand-entered fields are judgements and are recorded as such,
so a later reader can disagree with one specifically rather than with the whole row.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="wearable",
    source_table="heart_rate_summary + heart_rate_intraday",
    index_rule=INDEX_RULE,
    outcome_type="continuous",
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_any_hr"]),
    patients_analysable=paired_7d,
    coverage_share=paired_7d / cohort_patients,
    horizon_days=int(WINDOWS["postop"][1]),
    derivation_priced_gib=(estimated_bytes / 2**30 if estimated_bytes >= 0 else None),
    informative_missingness="yes",
    leakage_risk="low",
    notes=("Resting value is derived, not published: the 5th percentile of minute-level "
           "readings on days with at least 180 minutes recorded. Coverage above is from the "
           "daily summary and is therefore an upper bound on the derived metric."),
)